# What is preprocessing
Data preprocessing transforms messy, raw data into a clean, structured format that machine learning algorithms can understand
1. Data Collection and Integration
    - MySQL
    - Files - excel , csv etc
2. Feature selection
3. Data Cleaning
    * missing values
    * duplicate records 
    * remove outliers
4. Feature Engineering
5. Data Transformation
6. Feature scaling
 

In [ ]:
# loading imp libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

## Data Collection

### From file

In [ ]:
# loading data file of house prices using file
import pandas as pd
df = pd.read_csv('house_price.csv')
print(df.head())

### From MySQL

In [ ]:
from sqlalchemy import create_engine

# create a connection to the mysql database using sqlalchemy
host = 'localhost'          # set your details here
database = 'db'
user = 'root'
password = '1234'

engine = create_engine(f'mysql+mysqlconnector://{user}:{password}@{host}/{database}')

In [ ]:
pd.read_sql('SELECT * FROM house_price',engine)

In [ ]:
df.info()

In [ ]:
df.describe().round(2)

## Detecting missing values

In [ ]:
# detecting missing values in the dataset
df.isnull().sum()

## Remove duplicates

In [ ]:
# removing duplicate rows from the dataset
df = df.drop_duplicates()

# remove outliers

In [ ]:
# Use IQR Method


## Feature Engineering
Feature engineering is the process of using domain knowledge and statistical techniques to transform raw data into meaningful inputs (features) for machine learning models. Its goal is to highlight patterns and simplify data so algorithms can learn more effectively and make highly accurate predictions.

### Adding columns like
- total rooms
- bathroom per bedroom
- total area of all floors

In [ ]:
df.head()

## Feature Transformation
Transformation adjusts features to improve model learning:

- **Normalization & Scaling**: Adjust the range of features for consistency.
- **Encoding**: Converts categorical data to numerical form i.e one-hot encoding.
- **Mathematical** transformations: Like logarithmic transformations for skewed data.

### Scaling
- Min Max Scaler
- Standard scaler
- Roburst scaler

In [ ]:
df.area.describe().round(2)

In [ ]:
scaler = sklearn.preprocessing.MinMaxScaler()

scaled_data = pd.DataFrame(scaler.fit_transform(df.area.values.reshape(-1,1)))
print(scaled_data.head())
print(scaled_data.describe().round(2))

In [ ]:
scaler = sklearn.preprocessing.StandardScaler()

scaled_data = pd.DataFrame(scaler.fit_transform(df.area.values.reshape(-1,1)))
print(scaled_data.head())
print(scaled_data.describe().round(2))

In [ ]:
scaler = sklearn.preprocessing.RobustScaler()

scaled_data = pd.DataFrame(scaler.fit_transform(df.area.values.reshape(-1,1)))
print(scaled_data.head())
print(scaled_data.describe().round(2))

### Encoding
is the process of converting raw, categorical, or non-numeric data into a mathematical format that models can interpret.
- Lable encoding
- One Hot encoding

In [ ]:
df.select_dtypes(include=['object']).head()

**Label encoder** can only handle one column at a time 

In [ ]:
encoder = sklearn.preprocessing.LabelEncoder()

encoder.fit_transform(df.furnishingstatus)[:10]

So instead of that we can use ordinal encoder

In [ ]:
encoder = sklearn.preprocessing.OrdinalEncoder()
categorical_features = df.select_dtypes(include=['object']).columns

encoded_data = encoder.fit_transform(df[categorical_features])

In [ ]:
df[categorical_features]  = encoded_data  

In [ ]:
df.head()

**One-hot encoding** 

is a data preprocessing technique that converts categorical data (like words, labels, or text) into a binary numerical format so machine learning algorithms can understand it. It creates a separate new column for every unique category in a feature, filling the correct category's column with a 1 ("hot") and all other columns with a 0 ("cold").

In [43]:
df = pd.read_csv('house_price.csv')

In [73]:
encoder = sklearn.preprocessing.OneHotEncoder(sparse_output=False,drop = "first", handle_unknown='ignore')

encoded_data = encoder.fit_transform(df[categorical_features])

encoded_data

array([[1., 0., 0., ..., 1., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.],
       [1., 0., 1., ..., 1., 1., 0.],
       ...,
       [1., 0., 0., ..., 0., 0., 1.],
       [0., 0., 0., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 1.]])

**Dummy varicable**

In [74]:
pd.get_dummies(df[categorical_features], drop_first=False, dtype=int)

,mainroad_no,mainroad_yes,guestroom_no,guestroom_yes,basement_no,basement_yes,hotwaterheating_no,hotwaterheating_yes,airconditioning_no,airconditioning_yes,prefarea_no,prefarea_yes,furnishingstatus_furnished,furnishingstatus_semi-furnished,furnishingstatus_unfurnished
0,0,1,1,0,1,0,1,0,0,1,0,1,1,0,0
1,0,1,1,0,1,0,1,0,0,1,1,0,1,0,0
2,0,1,1,0,0,1,1,0,1,0,0,1,0,1,0
3,0,1,1,0,0,1,1,0,0,1,0,1,1,0,0
4,0,1,0,1,0,1,1,0,0,1,1,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
540,0,1,1,0,0,1,1,0,1,0,1,0,0,0,1
541,1,0,1,0,1,0,1,0,1,0,1,0,0,1,0
542,0,1,1,0,1,0,1,0,1,0,1,0,0,0,1
543,1,0,1,0,1,0,1,0,1,0,1,0,1,0,0


## ColumnTransformer
Every column needs to be transformed differently And we cannot Kee track of which column need to be transformed because some need scaling and other one hot so to keep track of all of them and avoide data leakages we use column transformer to keep track of all the transformation of the data 

To do this you will first need two lists one with a numerical feature name and one with a categorical features name 

In [ ]:
preprocessor = sklearn.compose.ColumnTransformer(
    transformers=["scaler", sklearn.preprocessing.StandardScaler(), ["area", "bedrooms"]],
    remainder="passthrough"
)